# PINN Simulation with DeepIRI ZepGPU

This notebook demonstrates Physics-Informed Neural Networks (PINNs) solving differential equations.

In [ ]:
import numpy as np
from deepiri_zepgpu import submit_task

## Define ODE Physics

In [ ]:
def lotka_volterra(t, y, alpha=1.0, beta=0.1, gamma=1.5, delta=0.075):
    """Lotka-Volterra predator-prey dynamics."""
    x, p = y
    dxdt = alpha * x - beta * x * p
    dpdt = delta * x * p - gamma * p
    return np.array([dxdt, dpdt])

def rk4_step(f, t, y, dt, **kwargs):
    """4th order Runge-Kutta integration step."""
    k1 = f(t, y, **kwargs)
    k2 = f(t + dt/2, y + dt*k1/2, **kwargs)
    k3 = f(t + dt/2, y + dt*k2/2, **kwargs)
    k4 = f(t + dt, y + dt*k3, **kwargs)
    return y + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

## Run Simulation on GPU

In [ ]:
def simulate_lotka_volterra(y0, t_span, num_steps):
    """Simulate Lotka-Volterra system."""
    t0, tf = t_span
    dt = (tf - t0) / num_steps
    
    y = np.array(y0, dtype=np.float64)
    trajectory = [y.copy()]
    
    for _ in range(num_steps):
        y = rk4_step(lotka_volterra, t0, y, dt)
        trajectory.append(y.copy())
    
    return np.array(trajectory)

# Initial conditions: [prey, predator]
y0 = [40, 9]  # x=40 prey, p=9 predators
t_span = (0, 20)  # 20 time units
num_steps = 2000

# Submit simulation
task_id = submit_task(
    simulate_lotka_volterra,
    y0, t_span, num_steps,
    gpu_memory_mb=1024
)
print(f"Simulation submitted: {task_id}")

## Analyze Results

In [ ]:
# Wait for result
trajectory = submit_task(
    simulate_lotka_volterra, y0, t_span, num_steps,
    wait=True
)

print(f"Trajectory shape: {trajectory.shape}")
print(f"Final prey population: {trajectory[-1, 0]:.2f}")
print(f"Final predator population: {trajectory[-1, 1]:.2f}")